# 🎮 Steam Games Analysis — El Efecto Indie

## Objetivo
Análisis exploratorio de +27.000 juegos de Steam (1997–2019) para identificar 
tendencias en géneros, valoraciones y precios.

## Preguntas clave
- ¿Qué géneros dominan el catálogo?
- ¿Los juegos gratuitos tienen mejor valoración que los de pago?
- ¿Cómo evolucionó la calidad promedio a lo largo del tiempo?
- ¿Qué desarrolladores publican más y con mejor reputación?

## Dataset
- **Fuente:** Kaggle — Steam Store Games
- **Registros:** 27.075 juegos
- **Columnas:** 18 variables (nombre, género, precio, valoraciones, plataforma, etc.)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset principal
df = pd.read_csv('steam.csv')
print('Filas y columnas:', df.shape)
df.head()

Filas y columnas: (27075, 18)


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


In [2]:
# Ver cuántos valores faltantes hay en cada columna
df.isnull().sum()


appid                0
name                 0
release_date         0
english              0
developer            1
publisher           14
platforms            0
required_age         0
categories           0
genres               0
steamspy_tags        0
achievements         0
positive_ratings     0
negative_ratings     0
average_playtime     0
median_playtime      0
owners               0
price                0
dtype: int64

## 2. Transformaciones

- Nulos en `developer` y `publisher` reemplazados por "Desconocido"
- `release_date` convertida a formato datetime y extraído el año
- Nueva columna `positive_pct`: porcentaje de valoraciones positivas sobre el total

In [3]:
# Rellenar los valores faltantes con "Desconocido"
df['developer'] = df['developer'].fillna('Desconocido')
df['publisher'] = df['publisher'].fillna('Desconocido')

# Convertir release_date a formato fecha
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Extraer el año en una columna nueva
df['release_year'] = df['release_date'].dt.year

# Calcular porcentaje de valoraciones positivas
df['total_ratings'] = df['positive_ratings'] + df['negative_ratings']
df['positive_pct'] = (df['positive_ratings'] / df['total_ratings'] * 100).round(2)

print('Listo! Nuevas columnas agregadas.')
df[['name', 'release_year', 'total_ratings', 'positive_pct']].head()


Listo! Nuevas columnas agregadas.


,name,release_year,total_ratings,positive_pct
0,Counter-Strike,2000,127873,97.39
1,Team Fortress Classic,1999,3951,83.98
2,Day of Defeat,2003,3814,89.56
3,Deathmatch Classic,2001,1540,82.66
4,Half-Life: Opposing Force,1999,5538,94.80


## 3. Exportación

Se eliminan duplicados por `appid` y se exporta el dataset limpio a CSV y SQLite para el análisis en SQL.

In [4]:
# Eliminar duplicados
df = df.drop_duplicates(subset='appid')

# Guardar el dataset limpio
df.to_csv('steam_clean.csv', index=False)

print('Dataset limpio guardado!')
print('Total de juegos:', len(df))

Dataset limpio guardado!
Total de juegos: 27075


In [5]:
import sqlite3

# Crear la base de datos
conn = sqlite3.connect('steam.db')
df.to_sql('games', conn, if_exists='replace', index=False)
conn.close()

print('Base de datos steam.db creada!')

Base de datos steam.db creada!
